# OpenSeek Track 3 — Phone-only sprint runner

Runs the full ICL annotation pipeline against a hosted Qwen3-4B endpoint and produces `submission.zip` for upload to flagos.io. Works entirely from a phone browser — no PC needed.

**Prerequisites**
1. Your `Opentract` repo is **public** on github.com. (Settings → General → Change repository visibility.) The notebook clones it.
2. You have a Qwen3-4B endpoint API key — recommended: [OpenRouter](https://openrouter.ai) free tier on `qwen/qwen3-4b:free` (or paid `qwen/qwen3-4b`).
3. **Save the API key as a Kaggle Secret** named `OPENROUTER_API_KEY` (Add-ons → Secrets → Add a new secret). This keeps the key out of the notebook.
4. Notebook settings: **Accelerator = None** (CPU is fine; the GPU work happens on the API side), **Internet = on**.

Total runtime: ~1–3 hours with 8-way concurrency. Kaggle sessions last 12h, so plenty of room.

---

In [ ]:
# 1. Configuration -- edit these three values for your setup.

REPO_URL    = 'https://github.com/Eienel/Opentract'          # your public fork
REPO_BRANCH = 'claude/flagos-hackathon-strategy-MeA51'

# Pick one of: OpenRouter (free or paid), DashScope, Together AI, etc.
BASE_URL    = 'https://openrouter.ai/api/v1'
MODEL       = 'qwen/qwen3-4b:free'   # try the :free variant first; fall back to 'qwen/qwen3-4b' if rate-limited

# Knobs
CONCURRENCY  = 8       # parallel in-flight requests; OpenRouter free is rate-limited, raise carefully
MAX_DEMO_TOK = 28000   # leave ~4K headroom for the system + query + completion
STRATEGY     = 'first_n'  # 'first_n' for the first leaderboard submission; 'similarity' once you want to tune
LIMIT_TESTS  = None    # set to e.g. 5 for a smoke run; None means full eval

In [ ]:
# 2. Clone the repo and install dependencies.

import os, subprocess, sys

REPO_DIR = '/kaggle/working/Opentract'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('repo + deps ready in', REPO_DIR)

In [ ]:
# 3. Fetch the 8 official competition datasets (~10 MB).

subprocess.check_call([sys.executable, 'openseek/scripts/fetch_data.py'])

In [ ]:
# 4. Wire up the API key from Kaggle Secrets and export config to env vars.

from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
os.environ['OPENAI_API_KEY']  = secret
os.environ['OPENAI_BASE_URL'] = BASE_URL
os.environ['OPENAI_MODEL']    = MODEL
print('base_url =', BASE_URL)
print('model    =', MODEL)
print('api_key  =', 'set' if secret else 'MISSING')

In [ ]:
# 5. Connectivity smoke test -- one tiny call, ~2s. Run this BEFORE the long batch.

subprocess.check_call([sys.executable, 'openseek/scripts/check_endpoint.py'])

In [ ]:
# 6. Quick smoke run on a few samples per task (~5 mins). Verifies the full path works end-to-end.

subprocess.check_call([
    sys.executable, 'openseek/scripts/run_all.py',
    '--strategy', STRATEGY,
    '--max-demo-tokens', str(MAX_DEMO_TOK),
    '--concurrency', str(CONCURRENCY),
    '--limit-tests', '5',
    '--out-dir', '/kaggle/working/run-smoke',
    '--submission', '/kaggle/working/submission-smoke.zip',
])

# Inspect a few predictions
import json
for tid in range(1, 9):
    with open(f'/kaggle/working/run-smoke/openseek-{tid}-v1.jsonl') as fh:
        first = fh.readline().strip()
    print(f'task {tid}: {first[:120]}')

In [ ]:
# 7. FULL RUN -- all 8 tasks, all 3666 test samples. ~1-3 hours with concurrency=8.

args = [
    sys.executable, 'openseek/scripts/run_all.py',
    '--strategy', STRATEGY,
    '--max-demo-tokens', str(MAX_DEMO_TOK),
    '--concurrency', str(CONCURRENCY),
    '--out-dir', '/kaggle/working/run-full',
    '--submission', '/kaggle/working/submission.zip',
]
if LIMIT_TESTS is not None:
    args += ['--limit-tests', str(LIMIT_TESTS)]

subprocess.check_call(args)

In [ ]:
# 8. Done. The submission ZIP is at /kaggle/working/submission.zip.
# Open the Kaggle output panel (right sidebar) -> Output -> download submission.zip,
# then upload it on the flagos.io Submission tab.

import os, zipfile
sub = '/kaggle/working/submission.zip'
print(f'submission: {sub}  ({os.path.getsize(sub)/1024:.1f} KB)')
with zipfile.ZipFile(sub) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        # Count predictions per task
        with zf.open(name) as fh:
            n_lines = sum(1 for ln in fh if ln.strip())
        print(f'  {name}  {info.file_size} bytes  {n_lines} predictions')